### Structured output 

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain support multiple schema types and methods for enforcing structured output

### Pydantic

Pydantic models provide the richest feature set with field validation, desriptions, and nested structures

In [11]:

from langchain_ollama import ChatOllama

# ollama model
ollama_model = ChatOllama(model="qwen2.5:3b", temperature=0)
ollama_model


ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen2.5:3b', temperature=0.0)

In [12]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description = "The title of the movie")
    year:int = Field(description = "This year the movie was released")
    director:str = Field(description = "The director of the movie")
    rating:float = Field(description = "The movies rating out of 10")

In [13]:
ollama_model_with_structure = ollama_model.with_structured_output(Movie)

In [14]:
ollama_model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen2.5:3b', temperature=0.0), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [15]:
ollama_model.invoke("Provide details about the moview Inception")

AIMessage(content='"Inception" is a science fiction film directed by Christopher Nolan and released in 2010. The movie is a complex narrative that explores themes of dreams, reality, and the nature of consciousness. Here are some key details about the film:\n\n### Plot Overview\nThe story revolves around Dom Cobb (Leonardo DiCaprio), a skilled thief who specializes in a technique called "extracting," where he enters a person\'s dreams to steal their secrets. Cobb is now in a coma and is being used as a pawn in a corporate espionage scheme orchestrated by his former partner, Arthur (Joseph Gordon-Levitt). The corporation, led by Robert Fischer (Al Pacino), wants to steal a corporate secret from the mind of a wealthy businessman, Mr. Hyde (Tom Hardy).\n\n### Key Characters\n- **Dom Cobb**: The main character, a skilled thief who can enter people\'s dreams.\n- **Arthur**: Cobb\'s former partner, now working for the corporation.\n- **Robert Fischer**: The head of the corporation, who wants

In [16]:
response = ollama_model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output Alongside Parsed Structure

In [17]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director pf the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

ollama_model_with_structure = ollama_model.with_structured_output(Movie, include_raw=True)

response = ollama_model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='{ "title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8 }', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-09-28T07:04:30.9864392Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11731744700, 'load_duration': 7008766800, 'prompt_eval_count': 36, 'prompt_eval_duration': 717228000, 'eval_count': 34, 'eval_duration': 3991226000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a0e6d4-2a32-7b13-a601-ebedf901171a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 34, 'total_tokens': 70}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [18]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description = "Budget in millions USD")

ollama_model_with_structure = ollama_model.with_structured_output(MovieDetails)

response = ollama_model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[], genres=['Action', 'Adventure', 'Sci-Fi'], budget=140000000.00000003)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you dont need runtime validation.

In [19]:
# Creating schema
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[str, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movie's rating out of 10"]

In [20]:
ollama_model_withtypedict = ollama_model.with_structured_output(MovieDict)
response = ollama_model_withtypedict.invoke("Please provide the deatils of the movi avengers")
response

{'title': 'Avengers',
 'year': '2012',
 'director': 'Joss Whedon',
 'rating': 'PG-13'}

### DataClasses

A data class is a class typically containig mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [22]:

from langchain_ollama import ChatOllama

# ollama model
ollama_model_agent = ChatOllama(model="qwen3:4b", temperature=0)
ollama_model_agent


ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen3:4b', temperature=0.0)

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=ollama_model_agent,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4564"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4564', additional_kwargs={}, response_metadata={}, id='e923e462-e7f2-4a5f-9291-932d476620f3'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-28T07:46:42.7042754Z', 'done': True, 'done_reason': 'stop', 'total_duration': 110777145500, 'load_duration': 8950167600, 'prompt_eval_count': 196, 'prompt_eval_duration': 3194509000, 'eval_count': 710, 'eval_duration': 98571223000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0e6f9-48d3-7db0-97b5-40ceebbd9aab-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4564'}, 'id': 'fc8ed768-9a9a-461d-bad9-3ac42d387ad2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 710, 'total_tokens': 906}),
  ToolMessage(content="Returning structur

In [25]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4564')

In [29]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact Information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=ollama_model_agent,
    # tools = tools,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4564"
    }]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4564'}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=ollama_model_agent,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4564"
    }]
})

result["structured_response"]